# 🎓 **Complete PLSR Tutorial with ALL Essential Plots**

## **From MLR to PLS2: Comprehensive Visualization & Interpretation**

---

### 📚 **Course Information**
- **Course:** TTK4260 - Multivariate Data Analysis and ML
- **Instructor:** Adil Rasheed, NTNU
- **Version:** Complete Edition with ~35 Plots

---

## 🎯 **What's New in This Version**

This **COMPLETE** tutorial includes ALL essential PLSR plots:

### ✅ **Interpretation Plots:**
- Correlation loadings (industry standard)
- Correlation circle plots
- Bi-plots (scores + loadings)
- 3D score visualization
- VIP scores

### ✅ **Diagnostic Plots:**
- Hotelling T² (outlier detection)
- Q residuals / DModX (model fit)
- Williams plot (applicability domain)
- Q-Q plot (normality check)
- Cook's distance (influential points)
- Residual diagnostics

### ✅ **Validation Plots:**
- R² train vs CV (overfitting)
- Explained variance (X and Y)
- Inner relation (PLS validity)
- Prediction intervals
- Per-fold CV analysis

### ✅ **Comparison Plots:**
- All methods side-by-side
- Performance metrics
- Coefficient evolution

**Total: ~35 interactive Plotly visualizations!**

---

## 📖 **Learning Objectives**

By the end, you will be able to:

1. ✅ Understand multicollinearity and why OLS fails
2. ✅ Apply PCR as unsupervised dimensionality reduction
3. ✅ Implement PLSR for supervised regression
4. ✅ **Interpret correlation loadings and circles** (NEW!)
5. ✅ **Detect outliers using T² and Q statistics** (NEW!)
6. ✅ **Assess prediction reliability with Williams plot** (NEW!)
7. ✅ **Validate model assumptions** (NEW!)
8. ✅ Extend to PLS2 for multiple responses
9. ✅ Choose the right method for your data

---

**⏱️ Estimated Time:** 3-4 hours (comprehensive)

Let's begin! 🚀

---

## **Section 0: Setup and Configuration**

In [51]:
# Core packages
import numpy as np
import pandas as pd
from scipy import stats

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold
from sklearn.metrics import mean_squared_error, r2_score

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Configuration
import warnings
warnings.filterwarnings('ignore')
pio.templates.default = "plotly_white"
np.random.seed(42)

print("✅ All packages loaded!")
print(f"NumPy: {np.__version__}")
print(f"Scikit-learn: {__import__('sklearn').__version__}")
print(f"Plotly: {__import__('plotly').__version__}")


✅ All packages loaded!
NumPy: 2.3.5
Scikit-learn: 1.7.2
Plotly: 6.3.0


---

## **Section 1: Data Loading and Exploration**

### **Understanding the Tecator Dataset**

**What is it?**
The Tecator dataset contains Near-Infrared (NIR) spectroscopy measurements of 215 finely chopped meat samples. Each sample was analyzed using an **Infratec Food and Feed Analyzer** to measure absorbance at 100 wavelengths between **850-1050 nm**.

**Why NIR Spectroscopy?**
- **C-H bonds** (fats) absorb strongly at ~930 nm
- **O-H bonds** (water/protein) show combinations near 980 nm
- Fat content creates **systematic but subtle** spectral patterns

**The Challenge:**
Traditional wet chemistry analysis for fat content is:
- Time-consuming (hours)
- Expensive (requires chemicals and trained personnel)
- Destructive (sample is lost)

**NIR spectroscopy offers:**
- ✅ Instant results (seconds)
- ✅ Non-destructive
- ✅ Cost-effective
- ❌ But requires sophisticated modeling due to **extreme multicollinearity** (adjacent wavelengths are 98-99% correlated!)

**The Problem:**
With 100 highly correlated predictors and only 215 samples, standard Multiple Linear Regression (MLR) becomes numerically unstable. This makes PLSR the method of choice in chemometrics.

**Real-World Impact:**
- Used in meat processing plants for quality control
- Enables real-time fat content monitoring
- Part of the food safety and labeling standards

---

### 1.1 Load Tecator Dataset

In [52]:
from skfda.datasets import fetch_tecator
from sklearn.model_selection import train_test_split

X_fda, y_fat = fetch_tecator(return_X_y=True)
X = X_fda.data_matrix.squeeze()
# Tecator dataset returns multiple responses: Fat, Moisture, Protein
Y_all = y_fat if y_fat.ndim > 1 else y_fat.reshape(-1, 1)
y = Y_all[:, 0]  # Fat content for single-response analysis
wavelengths = np.linspace(850, 1050, X.shape[1])

print("="*70)
print("TECATOR DATASET - MEAT QUALITY ANALYSIS")
print("="*70)
print(f"\nData dimensions:")
print(f"  X shape: {X.shape} (215 samples × 100 wavelengths)")
print(f"  Y shape: {Y_all.shape} (Fat, Moisture, Protein in %)")
print(f"  y (Fat) shape: {y.shape}")
print(f"\nWavelength range: {wavelengths.min():.0f}-{wavelengths.max():.0f} nm (Near-Infrared)")
print(f"\nFat content statistics:")
print(f"  Mean: {y.mean():.2f}% (typical for finely chopped meat)")
print(f"  Range: [{y.min():.2f}%, {y.max():.2f}%]")
print(f"  Std: {y.std():.2f}% (considerable variation)")
print(f"\n📊 This dataset is IDEAL for PLSR because:")
print(f"   • Many correlated predictors (100 wavelengths)")
print(f"   • Few samples (215)")
print(f"   • Continuous response (fat %)")

print(f"   • High multicollinearity (adjacent wavelengths ~99% correlated)")

TECATOR DATASET - MEAT QUALITY ANALYSIS

Data dimensions:
  X shape: (215, 100) (215 samples × 100 wavelengths)
  Y shape: (215, 3) (Fat, Moisture, Protein in %)
  y (Fat) shape: (215,)

Wavelength range: 850-1050 nm (Near-Infrared)

Fat content statistics:
  Mean: 18.14% (typical for finely chopped meat)
  Range: [0.90%, 49.10%]
  Std: 12.71% (considerable variation)

📊 This dataset is IDEAL for PLSR because:
   • Many correlated predictors (100 wavelengths)
   • Few samples (215)
   • Continuous response (fat %)
   • High multicollinearity (adjacent wavelengths ~99% correlated)


---

### **1.2 Train-Test Split (CRITICAL STEP!)**

**⚠️ Why split before ANY analysis?**
- Prevents **data leakage** (test set must remain truly unseen)
- Provides **honest performance estimates**
- Simulates real-world deployment (new samples)

We'll use:
- **80% training** (172 samples) - for model building and cross-validation
- **20% test** (43 samples) - ONLY touched at the very end for final evaluation

In [53]:
from sklearn.model_selection import train_test_split

# Perform train-test split
# Split BOTH fat (y) and all responses (Y_all) with same random_state
X_train, X_test, Y_all_train, Y_all_test = train_test_split(
    X, Y_all, test_size=0.2, random_state=42
)
y_train = Y_all_train[:, 0]  # Fat for single-response models
y_test = Y_all_test[:, 0]

print("="*70)
print("TRAIN-TEST SPLIT SUMMARY")
print("="*70)
print(f"\n📊 Training Set:")
print(f"   X_train shape: {X_train.shape} ({X_train.shape[0]} samples × {X_train.shape[1]} wavelengths)")
print(f"   Y_all_train shape: {Y_all_train.shape} (Fat, Moisture, Protein)")
print(f"   y_train (Fat) range: [{y_train.min():.2f}%, {y_train.max():.2f}%]")
print(f"   y_train mean: {y_train.mean():.2f}% ± {y_train.std():.2f}%")

print(f"\n🔒 Test Set (RESERVED for final evaluation):")
print(f"   X_test shape: {X_test.shape} ({X_test.shape[0]} samples × {X_test.shape[1]} wavelengths)")
print(f"   Y_all_test shape: {Y_all_test.shape}")
print(f"   y_test (Fat) range: [{y_test.min():.2f}%, {y_test.max():.2f}%]")
print(f"   y_test mean: {y_test.mean():.2f}% ± {y_test.std():.2f}%")


print(f"\n✅ Split complete! All subsequent analysis uses ONLY training data.")
print(f"   (Test set will be evaluated at the end)")

TRAIN-TEST SPLIT SUMMARY

📊 Training Set:
   X_train shape: (172, 100) (172 samples × 100 wavelengths)
   Y_all_train shape: (172, 3) (Fat, Moisture, Protein)
   y_train (Fat) range: [0.90%, 49.10%]
   y_train mean: 18.05% ± 12.59%

🔒 Test Set (RESERVED for final evaluation):
   X_test shape: (43, 100) (43 samples × 100 wavelengths)
   Y_all_test shape: (43, 3)
   y_test (Fat) range: [2.00%, 47.80%]
   y_test mean: 18.52% ± 13.17%

✅ Split complete! All subsequent analysis uses ONLY training data.
   (Test set will be evaluated at the end)


In [54]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Box Plot - Outlier Detection", "Histogram - Distribution"))

# Box plot (using TRAINING data only)
fig.add_trace(go.Box(y=y_train, name='Fat', marker_color='lightblue',
    boxmean='sd'), row=1, col=1)

# Histogram
fig.add_trace(go.Histogram(x=y_train, nbinsx=20, name='Fat',
    marker_color='lightcoral'), row=1, col=2)

fig.update_xaxes(title_text="", row=1, col=1)
fig.update_yaxes(title_text="Fat Content (%)", row=1, col=1)
fig.update_xaxes(title_text="Fat Content (%)", row=1, col=2)
fig.update_yaxes(title_text="Frequency", row=1, col=2)

fig.update_layout(title="Response Variable Analysis (Training Set)", width=1200, height=400,
    showlegend=False)
fig.show()

# Normality test
shapiro_stat, shapiro_p = stats.shapiro(y_train)
print("\n" + "="*70)
print("DISTRIBUTION ANALYSIS")
print("="*70)
print(f"\nShapiro-Wilk test: W={shapiro_stat:.4f}, p-value={shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("✅ Response appears normally distributed (p > 0.05)")
    print("   → Residual diagnostics will be valid")
else:
    print("⚠️  Response may deviate from normality (p < 0.05)")
    print("   → Consider checking residuals more carefully")

# Outlier detection using IQR method
Q1 = np.percentile(y_train, 25)
Q3 = np.percentile(y_train, 75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = y_train[(y_train < lower_bound) | (y_train > upper_bound)]

print(f"\nOutlier detection (IQR method):")
print(f"  IQR: {IQR:.2f}%")
print(f"  Bounds: [{lower_bound:.2f}%, {upper_bound:.2f}%]")
print(f"  Outliers: {len(outliers)} samples ({len(outliers)/len(y_train)*100:.1f}%)")
if len(outliers) > 0:
    print(f"  Values: {sorted(outliers)}")
    
print(f"\n📊 KEY INSIGHTS:")
print(f"   • Fat content ranges from {y_train.min():.1f}% to {y_train.max():.1f}%")
print(f"   • Most samples cluster around {y_train.mean():.1f}%")
print(f"   • Distribution is {'fairly symmetric' if shapiro_p > 0.05 else 'slightly skewed'}")
print(f"   • {'Few' if len(outliers) < 3 else 'Some'} extreme values present")


DISTRIBUTION ANALYSIS

Shapiro-Wilk test: W=0.9032, p-value=0.0000
⚠️  Response may deviate from normality (p < 0.05)
   → Consider checking residuals more carefully

Outlier detection (IQR method):
  IQR: 20.23%
  Bounds: [-22.74%, 58.16%]
  Outliers: 0 samples (0.0%)

📊 KEY INSIGHTS:
   • Fat content ranges from 0.9% to 49.1%
   • Most samples cluster around 18.0%
   • Distribution is slightly skewed
   • Few extreme values present


### 1.4 NIR Spectra Visualization (Training Data)

**What are we looking at?**
- Each line represents one meat sample's NIR absorbance spectrum
- **X-axis:** Wavelength (850-1050 nm) - NIR region
- **Y-axis:** Absorbance (higher = more light absorbed)
- **Color:** Blue (low fat) → Red (high fat)

**What should we see?**
- Systematic differences between low-fat and high-fat samples
- Key absorption peaks at ~930 nm (C-H bonds in fat) and ~980 nm (O-H bonds)

In [55]:
# Plot spectra colored by fat content (TRAINING DATA ONLY)
fig = go.Figure()

# Use all training samples with transparency
for i in range(len(X_train)):
    fig.add_trace(go.Scatter(
        x=wavelengths, y=X_train[i],
        mode='lines',
        line=dict(width=1, color=f'rgba({int(255*y_train[i]/y_train.max())},0,{int(255*(1-y_train[i]/y_train.max()))},0.3)'),
        showlegend=False,
        hovertemplate=f'Sample {i}<br>Fat: {y_train[i]:.1f}%<extra></extra>'
    ))

# Mark key wavelengths
fig.add_vline(x=930, line_dash="dash", line_color="green", 
              annotation_text="~930nm: C-H (fat)", annotation_position="top")
fig.add_vline(x=980, line_dash="dash", line_color="orange",
              annotation_text="~980nm: O-H (water/protein)", annotation_position="top")

fig.update_layout(
    title="NIR Absorbance Spectra (Training Set, colored by fat: blue=low, red=high)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Absorbance",
    width=1000, height=500
)
fig.show()

print("\n" + "="*70)
print("SPECTRAL ANALYSIS")
print("="*70)
print(f"\n🔍 OBSERVATIONS:")
# Find wavelengths with highest variation
spectral_variation = np.std(X_train, axis=0)
max_var_idx = np.argmax(spectral_variation)
print(f"   • Highest variation at λ={wavelengths[max_var_idx]:.0f}nm")
print(f"     (likely related to fat content differences)")

# Average spectra for low vs high fat
low_fat_mask = y_train < np.percentile(y_train, 33)
high_fat_mask = y_train > np.percentile(y_train, 67)
avg_low_fat = X_train[low_fat_mask].mean(axis=0)
avg_high_fat = X_train[high_fat_mask].mean(axis=0)
diff_spectrum = avg_high_fat - avg_low_fat
max_diff_idx = np.argmax(np.abs(diff_spectrum))

print(f"   • Greatest difference (high-fat vs low-fat) at λ={wavelengths[max_diff_idx]:.0f}nm")
print(f"     Difference: {diff_spectrum[max_diff_idx]:+.4f} absorbance units")
print(f"\n💡 KEY INSIGHT:")
print(f"   Spectra show systematic patterns related to fat content,")
print(f"   but patterns are SUBTLE and spread across many wavelengths.")
print(f"   This is why multivariate methods (PLSR) are essential!")


SPECTRAL ANALYSIS

🔍 OBSERVATIONS:
   • Highest variation at λ=933nm
     (likely related to fat content differences)
   • Greatest difference (high-fat vs low-fat) at λ=931nm
     Difference: +0.6660 absorbance units

💡 KEY INSIGHT:
   Spectra show systematic patterns related to fat content,
   but patterns are SUBTLE and spread across many wavelengths.
   This is why multivariate methods (PLSR) are essential!


### 1.4 Multicollinearity Assessment

In [56]:
from numpy.linalg import cond

# Compute multicollinearity metrics on TRAINING data only
condition_number = cond(X_train)
corr_matrix_X = np.corrcoef(X_train.T)
avg_corr = (np.sum(corr_matrix_X) - len(corr_matrix_X)) / (len(corr_matrix_X)**2 - len(corr_matrix_X))

print("="*70)
print("MULTICOLLINEARITY ANALYSIS (Training Data)")
print("="*70)
print(f"\n⚠️  Condition number: {condition_number:.2e}")
print(f"    (Values > 30 indicate severe multicollinearity)")
print(f"    Average correlation: {avg_corr:.3f}")
print(f"    (Adjacent wavelengths ~98-99% correlated!)")

if condition_number > 30:
    print(f"\n🚨 SEVERE multicollinearity detected!")
    print(f"   • OLS regression coefficients will be UNSTABLE")
    print(f"   • Small changes in data → huge coefficient swings")
    print(f"   • This is WHY we need PLSR or PCR!")

print(f"\n💡 REAL-WORLD IMPACT:")
print(f"   In meat processing, this means OLS can't reliably identify")
print(f"   which specific wavelengths matter. PLSR solves this by")
print(f"   creating orthogonal components.")

# Correlation heatmap (sample every 5th wavelength for visualization)
sample_every = 5
X_sample = X_train[:, ::sample_every]
corr_sample = np.corrcoef(X_sample.T)

fig = go.Figure(data=go.Heatmap(
    z=corr_sample,
    x=[f'{int(wavelengths[i])}' for i in range(0, len(wavelengths), sample_every)],
    y=[f'{int(wavelengths[i])}' for i in range(0, len(wavelengths), sample_every)],
    colorscale='RdBu_r',
    zmid=0,
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title="Correlation Matrix (every 5th wavelength)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Wavelength (nm)",
    width=700, height=700
)
fig.show()

MULTICOLLINEARITY ANALYSIS (Training Data)

⚠️  Condition number: 2.05e+07
    (Values > 30 indicate severe multicollinearity)
    Average correlation: 0.986
    (Adjacent wavelengths ~98-99% correlated!)

🚨 SEVERE multicollinearity detected!
   • OLS regression coefficients will be UNSTABLE
   • Small changes in data → huge coefficient swings
   • This is WHY we need PLSR or PCR!

💡 REAL-WORLD IMPACT:
   In meat processing, this means OLS can't reliably identify
   which specific wavelengths matter. PLSR solves this by
   creating orthogonal components.


### 1.5 Wavelength-Response Correlations

In [57]:
# Compute correlations using TRAINING data only
correlations_with_fat = np.array([
    np.corrcoef(X_train[:, i], y_train)[0, 1] for i in range(X_train.shape[1])
])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wavelengths, y=correlations_with_fat,
    mode='lines', fill='tozeroy',
    line=dict(color='purple', width=2)
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")

fig.update_layout(
    title="Wavelength-Fat Correlation Profile (Training Set)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Correlation with Fat Content",
    width=1000, height=400
)
fig.show()

print("\n" + "="*70)
print("WAVELENGTH IMPORTANCE ANALYSIS")
print("="*70)

max_corr_idx = np.argmax(np.abs(correlations_with_fat))
top_5_idx = np.argsort(np.abs(correlations_with_fat))[-5:]

print(f"\n🎯 STRONGEST individual correlation: λ={wavelengths[max_corr_idx]:.0f}nm")
print(f"   Correlation: {correlations_with_fat[max_corr_idx]:+.3f}")

print("\nTop 5 most correlated wavelengths:")
for idx in top_5_idx[::-1]:
    print(f"  λ={wavelengths[idx]:.0f}nm: r={correlations_with_fat[idx]:+.3f}")

print(f"\n📊 INTERPRETATION:")
print(f"   • All wavelengths show SOME correlation with fat")
print(f"   • But NO single wavelength dominates (max r ≈ {np.max(np.abs(correlations_with_fat)):.2f})")
print(f"   • Fat signal is DISTRIBUTED across spectrum")
print(f"   • This confirms need for multivariate approach (PLSR)")

print(f"\n⚠️  NOTE: These are univariate correlations!")
print(f"   PLSR will find COMBINATIONS of wavelengths that predict better.")


WAVELENGTH IMPORTANCE ANALYSIS

🎯 STRONGEST individual correlation: λ=931nm
   Correlation: +0.536

Top 5 most correlated wavelengths:
  λ=931nm: r=+0.536
  λ=929nm: r=+0.534
  λ=933nm: r=+0.533
  λ=927nm: r=+0.530
  λ=935nm: r=+0.527

📊 INTERPRETATION:
   • All wavelengths show SOME correlation with fat
   • But NO single wavelength dominates (max r ≈ 0.54)
   • Fat signal is DISTRIBUTED across spectrum
   • This confirms need for multivariate approach (PLSR)

⚠️  NOTE: These are univariate correlations!
   PLSR will find COMBINATIONS of wavelengths that predict better.


---

## **Section 2: Ordinary Least Squares (OLS)**

### 2.1 Standardize and Fit

In [58]:
# ⚠️ CRITICAL: Standardize using TRAINING data only!
# Fit scalers on training set, then transform both train and test
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train_std = scaler_X.fit_transform(X_train)  # Fit on train
X_test_std = scaler_X.transform(X_test)  # Transform test (no fitting!)
y_train_std = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_std = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

print("="*70)
print("OLS REGRESSION (Training Set)")
print("="*70)
print(f"\n⚠️  WHY OLS FAILS HERE:")
print(f"   • 172 samples, 100 predictors → underdetermined system")
print(f"   • Condition number: {cond(X_train_std):.2e} (>> 30)")
print(f"   • Adjacent wavelengths 99% correlated")
print(f"   • Result: Unstable, oscillating coefficients")

# Fit OLS on TRAINING data
mlr = LinearRegression()
mlr.fit(X_train_std, y_train_std)
y_train_pred_mlr_std = mlr.predict(X_train_std)
y_train_pred_mlr = scaler_y.inverse_transform(y_train_pred_mlr_std.reshape(-1, 1)).ravel()

r2_mlr_train = r2_score(y_train, y_train_pred_mlr)
rmse_mlr_train = np.sqrt(mean_squared_error(y_train, y_train_pred_mlr))

print(f"\nTraining Performance:")
print(f"  R²: {r2_mlr_train:.4f}")
print(f"  RMSE: {rmse_mlr_train:.4f}%")
print(f"\n(Note: Good training scores don't guarantee good generalization!)")

coef_mlr = mlr.coef_
print(f"\nCoefficients:")
print(f"  Mean |coef|: {np.abs(coef_mlr).mean():.4f}")
print(f"  Max |coef|: {np.abs(coef_mlr).max():.4f}")
print(f"  Std: {coef_mlr.std():.4f}")

OLS REGRESSION (Training Set)

⚠️  WHY OLS FAILS HERE:
   • 172 samples, 100 predictors → underdetermined system
   • Condition number: 3.15e+06 (>> 30)
   • Adjacent wavelengths 99% correlated
   • Result: Unstable, oscillating coefficients

Training Performance:
  R²: 0.9965
  RMSE: 0.7464%

(Note: Good training scores don't guarantee good generalization!)

Coefficients:
  Mean |coef|: 632.3660
  Max |coef|: 2009.1594
  Std: 807.6762


### 2.2 OLS Coefficients (Extremely Noisy!)

In [59]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wavelengths, y=coef_mlr,
    mode='lines+markers',
    line=dict(color='red', width=2),
    marker=dict(size=3)
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="OLS Coefficients - UNSTABLE due to Multicollinearity",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Coefficient Value",
    width=1000, height=500
)
fig.show()

print("⚠️  Wild oscillations → numerical instability!")

⚠️  Wild oscillations → numerical instability!


### 2.3 Cross-Validation

In [60]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)

# Cross-validation on TRAINING set only
cv_scores_mlr = cross_val_score(mlr, X_train_std, y_train_std, cv=cv, 
                                 scoring='neg_mean_squared_error')
cv_rmse_mlr = np.sqrt(-cv_scores_mlr)
cv_r2_mlr = cross_val_score(mlr, X_train_std, y_train_std, cv=cv, scoring='r2')

y_train_pred_cv_mlr_std = cross_val_predict(mlr, X_train_std, y_train_std, cv=cv)
y_train_pred_cv_mlr = scaler_y.inverse_transform(y_train_pred_cv_mlr_std.reshape(-1, 1)).ravel()

print("="*70)
print("CROSS-VALIDATION RESULTS (10-fold on Training Set)")
print("="*70)
print(f"  CV R²: {cv_r2_mlr.mean():.4f} ± {cv_r2_mlr.std():.4f}")
print(f"  CV RMSE: {cv_rmse_mlr.mean():.4f} ± {cv_rmse_mlr.std():.4f}")

overfitting_gap = r2_mlr_train - cv_r2_mlr.mean()
if overfitting_gap > 0.1:
    print(f"\n🚨 SEVERE OVERFITTING DETECTED!")
    print(f"   Train R²: {r2_mlr_train:.4f}")
    print(f"   CV R²: {cv_r2_mlr.mean():.4f}")
    print(f"   Gap: {overfitting_gap:.3f}")
    print(f"\n   WHY? Multicollinearity allows model to 'memorize' noise!")
else:
    print(f"\n✅ Reasonable generalization (gap: {overfitting_gap:.3f})")

CROSS-VALIDATION RESULTS (10-fold on Training Set)
  CV R²: 0.9396 ± 0.0260
  CV RMSE: 0.2335 ± 0.0522

✅ Reasonable generalization (gap: 0.057)


### 2.4 OLS Diagnostics

In [61]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Training Set", "Cross-Validation (Training Only)"))

# Training predictions
fig.add_trace(go.Scatter(x=y_train, y=y_train_pred_mlr, mode='markers',
    marker=dict(color='blue', size=6, opacity=0.6),
    name='Train'), row=1, col=1)

# CV predictions
fig.add_trace(go.Scatter(x=y_train, y=y_train_pred_cv_mlr, mode='markers',
    marker=dict(color='red', size=6, opacity=0.6),
    name='CV'), row=1, col=2)

# Perfect prediction lines
y_range = [y_train.min(), y_train.max()]
for col in [1, 2]:
    fig.add_trace(go.Scatter(x=y_range, y=y_range, mode='lines',
        line=dict(color='black', dash='dash'), showlegend=False), row=1, col=col)

fig.update_xaxes(title_text="Actual Fat (%)", row=1, col=1)
fig.update_xaxes(title_text="Actual Fat (%)", row=1, col=2)
fig.update_yaxes(title_text="Predicted (%)", row=1, col=1)
fig.update_yaxes(title_text="Predicted (%)", row=1, col=2)

fig.update_layout(title="OLS: Actual vs Predicted (Training Data Only)", 
                  width=1200, height=500, showlegend=False)
fig.show()

print("\n💡 WHAT THIS SHOWS:")
print("   Left: Model fits training data reasonably well")
print("   Right: CV predictions worse → model is overfitting to noise")
print("   → This is why we need PCR or PLSR!")


💡 WHAT THIS SHOWS:
   Left: Model fits training data reasonably well
   Right: CV predictions worse → model is overfitting to noise
   → This is why we need PCR or PLSR!


---

## **Section 3: Principal Component Regression (PCR)**

### 3.1 PCA Analysis

In [62]:
pca_full = PCA(n_components=None)
T_train_full = pca_full.fit_transform(X_train_std)  # Fit on training data only!
P_full = pca_full.components_.T

explained_var_ratio = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var_ratio)

print("="*70)
print("PRINCIPAL COMPONENT ANALYSIS (Training Data)")
print("="*70)
print(f"\nVariance explained (on training set):")
print(f"  PC1: {explained_var_ratio[0]*100:.2f}%")
print(f"  PC1-2: {cumulative_var[1]*100:.2f}%")
print(f"  PC1-5: {cumulative_var[4]*100:.2f}%")
print(f"  PC1-10: {cumulative_var[9]*100:.2f}%")

n_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f"\n  Components for 95% variance: {n_95}")
print(f"\n💡 FOR THIS TECATOR DATA:")
print(f"   • PC1 captures {explained_var_ratio[0]*100:.1f}% of spectral variation")
print(f"   • {n_95} components capture 95% of information from 100 wavelengths!")
print(f"   • Dramatic compression possible due to wavelength correlations")

PRINCIPAL COMPONENT ANALYSIS (Training Data)

Variance explained (on training set):
  PC1: 98.63%
  PC1-2: 99.60%
  PC1-5: 100.00%
  PC1-10: 100.00%

  Components for 95% variance: 1

💡 FOR THIS TECATOR DATA:
   • PC1 captures 98.6% of spectral variation
   • 1 components capture 95% of information from 100 wavelengths!
   • Dramatic compression possible due to wavelength correlations


### 3.2 Scree Plot

In [63]:
n_show = 30
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Scree Plot", "Cumulative Variance"))

fig.add_trace(go.Bar(x=np.arange(1, n_show+1),
    y=explained_var_ratio[:n_show]*100), row=1, col=1)

fig.add_trace(go.Scatter(x=np.arange(1, n_show+1),
    y=cumulative_var[:n_show]*100, mode='lines+markers',
    line=dict(color='red', width=2)), row=1, col=2)

fig.add_hline(y=95, line_dash="dash", line_color="green", row=1, col=2)

fig.update_xaxes(title_text="PC", row=1, col=1)
fig.update_xaxes(title_text="# Components", row=1, col=2)
fig.update_yaxes(title_text="Variance (%)", row=1, col=1)
fig.update_yaxes(title_text="Cumulative (%)", row=1, col=2)

fig.update_layout(title="PCA Variance Explained", width=1200, height=500,
    showlegend=False)
fig.show()

### 3.3 PC Correlation with Response

In [64]:
n_pcs = 20
# Compute correlations using TRAINING data only
pc_corr = [np.corrcoef(T_train_full[:, i], y_train)[0, 1] for i in range(n_pcs)]

fig = go.Figure()
fig.add_trace(go.Bar(x=np.arange(1, n_pcs+1), y=pc_corr,
    marker_color=['red' if abs(c)>0.5 else 'lightblue' for c in pc_corr]))
fig.add_hline(y=0, line_color="black")
fig.add_hline(y=0.5, line_dash="dash", line_color="green", 
              annotation_text="Strong correlation threshold")
fig.add_hline(y=-0.5, line_dash="dash", line_color="green")

fig.update_layout(
    title="PC Correlation with Fat Content (Training Set)",
    xaxis_title="Principal Component",
    yaxis_title="Correlation with Fat %",
    width=1000, height=500
)
fig.show()

print("\n" + "="*70)
print("PC-RESPONSE CORRELATION ANALYSIS")
print("="*70)
strongest_pc = np.argmax(np.abs(pc_corr))
print(f"\n🎯 STRONGEST predictor: PC{strongest_pc+1} (r={pc_corr[strongest_pc]:+.3f})")
print(f"\n⚠️  CRITICAL INSIGHT:")
print(f"   • PC1 explains {explained_var_ratio[0]*100:.0f}% of spectral variance")
print(f"   • But PC1 correlation with fat: r={pc_corr[0]:+.3f}")
print(f"   • High-variance PCs ≠ high-prediction PCs!")
print(f"   • This is why PLSR often outperforms PCR")


PC-RESPONSE CORRELATION ANALYSIS

🎯 STRONGEST predictor: PC3 (r=+0.561)

⚠️  CRITICAL INSIGHT:
   • PC1 explains 99% of spectral variance
   • But PC1 correlation with fat: r=+0.461
   • High-variance PCs ≠ high-prediction PCs!
   • This is why PLSR often outperforms PCR


### 3.4 PCR Cross-Validation

In [65]:
def fit_pcr(X_train, y_train, X_test, y_test, n_components):
    pca = PCA(n_components=n_components)
    T_train = pca.fit_transform(X_train)
    reg = LinearRegression()
    reg.fit(T_train, y_train)
    T_test = pca.transform(X_test)
    y_pred = reg.predict(T_test)
    return y_pred, pca, reg

max_components = 30
print(f"Running PCR CV on TRAINING set (max {max_components} components)...")

pcr_results = {}
for n_comp in range(1, max_components + 1):
    cv_scores = []
    # CV within training set only
    for train_idx, val_idx in cv.split(X_train_std):
        X_cv_train, X_cv_val = X_train_std[train_idx], X_train_std[val_idx]
        y_cv_train, y_cv_val = y_train_std[train_idx], y_train_std[val_idx]
        y_pred, _, _ = fit_pcr(X_cv_train, y_cv_train, X_cv_val, y_cv_val, n_comp)
        cv_scores.append(mean_squared_error(y_cv_val, y_pred))
    pcr_results[n_comp] = {
        'cv_rmse_mean': np.sqrt(np.mean(cv_scores)),
        'cv_rmse_std': np.std(np.sqrt(cv_scores))
    }

rmse_values_pcr = [pcr_results[k]['cv_rmse_mean'] for k in range(1, max_components+1)]
optimal_k_pcr = np.argmin(rmse_values_pcr) + 1

print(f"✅ Optimal: k={optimal_k_pcr} components")
print(f"   CV RMSE: {rmse_values_pcr[optimal_k_pcr-1]:.4f}")
print(f"\n💡 INTERPRETATION:")
print(f"   {optimal_k_pcr} PCs capture the fat-related signal while filtering noise")

Running PCR CV on TRAINING set (max 30 components)...
✅ Optimal: k=17 components
   CV RMSE: 0.2007

💡 INTERPRETATION:
   17 PCs capture the fat-related signal while filtering noise


### 3.5 PCR Model with Optimal Components

In [66]:
# Fit optimal PCR model on full TRAINING set
pca_optimal = PCA(n_components=optimal_k_pcr)
T_train_optimal = pca_optimal.fit_transform(X_train_std)  # Training scores
P_optimal = pca_optimal.components_.T

reg_pcr = LinearRegression()
reg_pcr.fit(T_train_optimal, y_train_std)
y_train_pred_pcr_std = reg_pcr.predict(T_train_optimal)
y_train_pred_pcr = scaler_y.inverse_transform(y_train_pred_pcr_std.reshape(-1, 1)).ravel()

# Map PC regression coefficients back to original wavelength space
theta_pcr = P_optimal @ reg_pcr.coef_
r2_pcr_train = r2_score(y_train, y_train_pred_pcr)

# CV predictions for training set
y_train_pred_pcr_cv_std = np.zeros(len(y_train_std))
for train_idx, val_idx in cv.split(X_train_std):
    X_cv_train, X_cv_val = X_train_std[train_idx], X_train_std[val_idx]
    y_cv_train, y_cv_val = y_train_std[train_idx], y_train_std[val_idx]
    y_pred, _, _ = fit_pcr(X_cv_train, y_cv_train, X_cv_val, y_cv_val, optimal_k_pcr)
    y_train_pred_pcr_cv_std[val_idx] = y_pred

y_train_pred_pcr_cv = scaler_y.inverse_transform(y_train_pred_pcr_cv_std.reshape(-1, 1)).ravel()
r2_pcr_cv = r2_score(y_train, y_train_pred_pcr_cv)
rmse_pcr_cv = np.sqrt(mean_squared_error(y_train, y_train_pred_pcr_cv))

print("="*70)
print(f"PRINCIPAL COMPONENT REGRESSION (k={optimal_k_pcr} components)")
print("="*70)
print(f"  Training R²: {r2_pcr_train:.4f}")
print(f"  CV R²: {r2_pcr_cv:.4f}")
print(f"  CV RMSE: {rmse_pcr_cv:.4f}%")
print(f"\n💡 KEY POINTS:")
print(f"   • PCR reduces 100 wavelengths to {optimal_k_pcr} orthogonal components")
print(f"   • Much better generalization than OLS (CV R² vs train R²)")
print(f"   • But: PCs optimized for variance, not fat prediction")

PRINCIPAL COMPONENT REGRESSION (k=17 components)
  Training R²: 0.9739
  CV R²: 0.9593
  CV RMSE: 2.5409%

💡 KEY POINTS:
   • PCR reduces 100 wavelengths to 17 orthogonal components
   • Much better generalization than OLS (CV R² vs train R²)
   • But: PCs optimized for variance, not fat prediction


---

## **Section 4: Partial Least Squares Regression (PLS)**

### 4.1 PLS Cross-Validation

In [67]:
print("="*70)
print("PLS CROSS-VALIDATION (Training Set)")
print("="*70)
print(f"Testing 1 to {max_components} components...")

pls_results = {}
for n_comp in range(1, max_components + 1):
    cv_scores = []
    # CV within training set only
    for train_idx, val_idx in cv.split(X_train_std):
        X_cv_train, X_cv_val = X_train_std[train_idx], X_train_std[val_idx]
        y_cv_train, y_cv_val = y_train_std[train_idx], y_train_std[val_idx]
        pls = PLSRegression(n_components=n_comp, scale=False)
        pls.fit(X_cv_train, y_cv_train)
        y_pred = pls.predict(X_cv_val).ravel()
        cv_scores.append(mean_squared_error(y_cv_val, y_pred))
    pls_results[n_comp] = {
        'cv_rmse_mean': np.sqrt(np.mean(cv_scores)),
        'cv_rmse_std': np.std(np.sqrt(cv_scores))
    }

rmse_values_pls = [pls_results[k]['cv_rmse_mean'] for k in range(1, max_components+1)]
optimal_k_pls = np.argmin(rmse_values_pls) + 1

print(f"\n✅ Optimal: {optimal_k_pls} latent variables")
print(f"   CV RMSE: {rmse_values_pls[optimal_k_pls-1]:.4f}")
print(f"\n💡 PLSR vs PCR:")
print(f"   PCR optimal: {optimal_k_pcr} components")
print(f"   PLS optimal: {optimal_k_pls} components")
if optimal_k_pls < optimal_k_pcr:
    print(f"   → PLSR is more efficient (finds fat signal in fewer dimensions)")

PLS CROSS-VALIDATION (Training Set)
Testing 1 to 30 components...

✅ Optimal: 13 latent variables
   CV RMSE: 0.1968

💡 PLSR vs PCR:
   PCR optimal: 17 components
   PLS optimal: 13 components
   → PLSR is more efficient (finds fat signal in fewer dimensions)


### 4.2 PCR vs PLS Comparison

In [68]:
fig = go.Figure()

k_values = list(range(1, max_components+1))

fig.add_trace(go.Scatter(x=k_values, y=rmse_values_pcr,
    mode='lines+markers', name='PCR', line=dict(color='blue', width=2)))

fig.add_trace(go.Scatter(x=k_values, y=rmse_values_pls,
    mode='lines+markers', name='PLS', line=dict(color='red', width=2)))

fig.add_vline(x=optimal_k_pcr, line_dash="dash", line_color="blue",
    annotation_text=f"PCR k={optimal_k_pcr}")
fig.add_vline(x=optimal_k_pls, line_dash="dash", line_color="red",
    annotation_text=f"PLS k={optimal_k_pls}")

fig.update_layout(
    title="Model Selection: PCR vs PLS",
    xaxis_title="Number of Components",
    yaxis_title="RMSE (CV)",
    width=1000, height=500
)
fig.show()

print(f"\nPCR: k={optimal_k_pcr}, RMSE={rmse_values_pcr[optimal_k_pcr-1]:.4f}")
print(f"PLS: k={optimal_k_pls}, RMSE={rmse_values_pls[optimal_k_pls-1]:.4f}")
if optimal_k_pls < optimal_k_pcr:
    print(f"✅ PLS uses {optimal_k_pcr-optimal_k_pls} FEWER components!")


PCR: k=17, RMSE=0.2007
PLS: k=13, RMSE=0.1968
✅ PLS uses 4 FEWER components!


### 4.3 Fit Optimal PLS Model

In [69]:
# Fit optimal PLS model on full TRAINING set
pls_optimal = PLSRegression(n_components=optimal_k_pls, scale=False)
pls_optimal.fit(X_train_std, y_train_std)

y_train_pred_pls_std = pls_optimal.predict(X_train_std).ravel()
y_train_pred_pls = scaler_y.inverse_transform(y_train_pred_pls_std.reshape(-1, 1)).ravel()
r2_pls_train = r2_score(y_train, y_train_pred_pls)

# CV predictions on training set
y_train_pred_pls_cv_std = np.zeros(len(y_train_std))
for train_idx, val_idx in cv.split(X_train_std):
    X_cv_train, X_cv_val = X_train_std[train_idx], X_train_std[val_idx]
    y_cv_train, y_cv_val = y_train_std[train_idx], y_train_std[val_idx]
    pls_cv = PLSRegression(n_components=optimal_k_pls, scale=False)
    pls_cv.fit(X_cv_train, y_cv_train)
    y_train_pred_pls_cv_std[val_idx] = pls_cv.predict(X_cv_val).ravel()

y_train_pred_pls_cv = scaler_y.inverse_transform(y_train_pred_pls_cv_std.reshape(-1, 1)).ravel()
r2_pls_cv = r2_score(y_train, y_train_pred_pls_cv)
rmse_pls_cv = np.sqrt(mean_squared_error(y_train, y_train_pred_pls_cv))

# Extract PLS components (from training data)
T_pls = pls_optimal.x_scores_  # Training set scores
W_pls = pls_optimal.x_weights_  # Weights
P_pls = pls_optimal.x_loadings_  # X loadings
Q_pls = pls_optimal.y_loadings_  # Y loadings
coef_pls = pls_optimal.coef_.ravel()  # Regression coefficients

print("="*70)
print(f"PARTIAL LEAST SQUARES REGRESSION (k={optimal_k_pls} components)")
print("="*70)
print(f"  Training R²: {r2_pls_train:.4f}")
print(f"  CV R²: {r2_pls_cv:.4f}")
print(f"  CV RMSE: {rmse_pls_cv:.4f}%")
print(f"\n💡 WHY PLSR WORKS FOR TECATOR DATA:")
print(f"   • Finds {optimal_k_pls} latent variables optimized for fat prediction")
print(f"   • Unlike PCR, components directly maximize covariance with response")
print(f"   • Handles multicollinearity while preserving predictive power")

print(f"\nComponent correlations with Y:")
for i in range(min(5, optimal_k_pls)):
    corr = np.corrcoef(T_pls[:, i], y_train_std)[0, 1]
    print(f"  Comp {i+1}: {corr:+.3f}")

PARTIAL LEAST SQUARES REGRESSION (k=13 components)
  Training R²: 0.9768
  CV R²: 0.9609
  CV RMSE: 2.4892%

💡 WHY PLSR WORKS FOR TECATOR DATA:
   • Finds 13 latent variables optimized for fat prediction
   • Unlike PCR, components directly maximize covariance with response
   • Handles multicollinearity while preserving predictive power

Component correlations with Y:
  Comp 1: +0.464
  Comp 2: +0.628
  Comp 3: +0.462
  Comp 4: -0.252
  Comp 5: +0.238


### 4.4 Standard PLS Score Plot

In [70]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=T_pls[:, 0],
    y=T_pls[:, 1] if optimal_k_pls > 1 else np.zeros(len(T_pls)),
    mode='markers',
    marker=dict(size=8, color=y_train, colorscale='Viridis',
        showscale=True, colorbar=dict(title="Fat (%)")),
    text=[f'Sample {i}: {y_train[i]:.1f}%' for i in range(len(y_train))],
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    title="PLS Score Plot (colored by Fat)",
    xaxis_title=f"Comp 1 (r={np.corrcoef(T_pls[:,0], y_train_std)[0,1]:.3f})",
    yaxis_title=f"Comp 2" + (f" (r={np.corrcoef(T_pls[:,1], y_train_std)[0,1]:.3f})" if optimal_k_pls>1 else ""),
    width=800, height=600
)
fig.show()

print("💡 Strong color gradient: PLS maximizes correlation with Y!")

💡 Strong color gradient: PLS maximizes correlation with Y!


### 4.5 ⭐ **NEW: 3D Score Plot**

In [71]:
if optimal_k_pls >= 3:
    fig = go.Figure(data=[go.Scatter3d(
        x=T_pls[:, 0],
        y=T_pls[:, 1],
        z=T_pls[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=y_train,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Fat (%)", x=1.1)
        ),
        text=[f'Sample {i}<br>Fat: {y_train[i]:.1f}%' for i in range(len(y_train))],
        hovertemplate='%{text}<extra></extra>'
    )])
    
    fig.update_layout(
        title="3D PLS Score Plot",
        scene=dict(
            xaxis_title="Component 1",
            yaxis_title="Component 2",
            zaxis_title="Component 3"
        ),
        width=800,
        height=800
    )
    fig.show()
    print("✅ 3D view shows sample clustering in latent space")
else:
    print("⚠️  Need ≥3 components for 3D plot")

✅ 3D view shows sample clustering in latent space


### 4.6 Standard Loadings (P)

In [72]:
fig = go.Figure()
n_comp_plot = min(4, optimal_k_pls)
colors = ['blue', 'red', 'green', 'orange']

for i in range(n_comp_plot):
    fig.add_trace(go.Scatter(
        x=wavelengths, y=P_pls[:, i],
        mode='lines', name=f'Comp {i+1}',
        line=dict(color=colors[i], width=2)
    ))

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="PLS Standard Loadings (P)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Loading",
    width=1000, height=500
)
fig.show()

### 4.7 ⭐⭐⭐ **NEW: Correlation Loadings** (INDUSTRY STANDARD)

**CRITICAL:** Correlation loadings are the STANDARD for PLS interpretation!
- Bounded [-1, +1] (easy to understand)
- Show correlation between variables and components
- Used by SIMCA, Unscrambler, all chemometrics software

In [73]:
# Compute correlation loadings using TRAINING data
correlation_loadings = np.zeros((X_train_std.shape[1], optimal_k_pls))

for i in range(optimal_k_pls):
    for j in range(X_train_std.shape[1]):
        correlation_loadings[j, i] = np.corrcoef(X_train_std[:, j], T_pls[:, i])[0, 1]

print("="*70)
print("CORRELATION LOADINGS (Training Set)")
print("="*70)
print(f"Shape: {correlation_loadings.shape}")
print(f"Range: [{correlation_loadings.min():.3f}, {correlation_loadings.max():.3f}]")
print("\n✅ These show wavelength-component correlations!")
print("💡 FOR TECATOR DATA:")
print(f"   Component 1: Shows dominant spectral pattern")
print(f"   High positive loading at wavelength λ means:")
print(f"   → Samples with high absorbance at λ have high component score")

# Plot correlation loadings
fig = go.Figure()

for i in range(min(4, optimal_k_pls)):
    fig.add_trace(go.Scatter(
        x=wavelengths,
        y=correlation_loadings[:, i],
        mode='lines',
        name=f'Component {i+1}',
        line=dict(color=colors[i], width=2)
    ))

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.add_hline(y=0.7, line_dash="dot", line_color="green",
              annotation_text="Strong (+)")
fig.add_hline(y=-0.7, line_dash="dot", line_color="green",
              annotation_text="Strong (-)")

fig.update_layout(
    title="⭐ PLS Correlation Loadings (RECOMMENDED for Interpretation)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Correlation with Component",
    yaxis_range=[-1, 1],
    width=1000,
    height=500,
    hovermode='x unified'
)

fig.show()

print("\n💡 Interpretation:")
print("  |r| > 0.7: Strong correlation")
print("  |r| < 0.3: Weak correlation")
print("  Sign: Positive/negative relationship")

CORRELATION LOADINGS (Training Set)
Shape: (100, 13)
Range: [-0.140, 0.999]

✅ These show wavelength-component correlations!
💡 FOR TECATOR DATA:
   Component 1: Shows dominant spectral pattern
   High positive loading at wavelength λ means:
   → Samples with high absorbance at λ have high component score



💡 Interpretation:
  |r| > 0.7: Strong correlation
  |r| < 0.3: Weak correlation
  Sign: Positive/negative relationship


### 4.8 ⭐⭐⭐ **NEW: Correlation Circle Plot**

The **correlation circle** is the STANDARD plot in chemometrics:
- Variables plotted on unit circle
- Distance from origin = importance
- Angle between variables = their correlation
- Standard in all professional software

In [74]:
if optimal_k_pls >= 2:
    fig = go.Figure()
    
    # Plot variables (every 5th for clarity)
    sample_every = 5
    for i in range(0, len(wavelengths), sample_every):
        # Arrow from origin
        fig.add_trace(go.Scatter(
            x=[0, correlation_loadings[i, 0]],
            y=[0, correlation_loadings[i, 1]],
            mode='lines',
            line=dict(color='lightblue', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))
        
        # Point at tip
        fig.add_trace(go.Scatter(
            x=[correlation_loadings[i, 0]],
            y=[correlation_loadings[i, 1]],
            mode='markers',
            marker=dict(size=4, color='blue'),
            showlegend=False,
            text=f'λ={wavelengths[i]:.0f}nm',
            hovertemplate='%{text}<br>r1=%{x:.3f}<br>r2=%{y:.3f}<extra></extra>'
        ))
    
    # Unit circle
    theta = np.linspace(0, 2*np.pi, 100)
    fig.add_trace(go.Scatter(
        x=np.cos(theta),
        y=np.sin(theta),
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        name='Unit circle'
    ))
    
    # Axes
    fig.add_hline(y=0, line_color="black", line_width=1)
    fig.add_vline(x=0, line_color="black", line_width=1)
    
    fig.update_layout(
        title="⭐ Correlation Circle Plot (Standard in Chemometrics)",
        xaxis_title="Correlation with Component 1",
        yaxis_title="Correlation with Component 2",
        width=700,
        height=700,
        xaxis=dict(scaleanchor="y", scaleratio=1, range=[-1.1, 1.1]),
        yaxis=dict(range=[-1.1, 1.1])
    )
    
    fig.show()
    
    print("\n💡 Interpretation:")
    print("  • Variables far from origin: well represented")
    print("  • Variables close together: positively correlated")
    print("  • Variables opposite: negatively correlated")
    print("  • Variables at 90°: uncorrelated")
else:
    print("⚠️  Need ≥2 components for correlation circle")


💡 Interpretation:
  • Variables far from origin: well represented
  • Variables close together: positively correlated
  • Variables opposite: negatively correlated
  • Variables at 90°: uncorrelated


### 4.9 VIP Scores

In [75]:
def compute_vip(pls_model):
    W = pls_model.x_weights_
    Q = pls_model.y_loadings_
    T = pls_model.x_scores_
    ss = np.sum(T**2, axis=0) * (Q**2).ravel()
    total_ss = np.sum(ss)
    p = W.shape[0]
    vip = np.sqrt(p * np.sum(ss * W**2, axis=1) / total_ss)
    return vip

vip_scores = compute_vip(pls_optimal)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wavelengths, y=vip_scores,
    mode='lines', fill='tozeroy',
    line=dict(color='darkblue', width=2)
))
fig.add_hline(y=1.0, line_dash="dash", line_color="red",
    annotation_text="VIP = 1 (threshold)")

fig.update_layout(
    title="Variable Importance in Projection (VIP)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="VIP Score",
    width=1000, height=500
)
fig.show()

important_vip = np.sum(vip_scores > 1.0)
top_vip_idx = np.argsort(vip_scores)[-5:][::-1]

print("\n" + "="*70)
print("VIP SCORE INTERPRETATION")
print("="*70)
print(f"\nWavelengths with VIP > 1: {important_vip}/{len(wavelengths)}")
print(f"  ({important_vip/len(wavelengths)*100:.1f}% of wavelengths are 'important')")

print(f"\nTop 5 most important wavelengths for fat prediction:")
for idx in top_vip_idx:
    print(f"  λ={wavelengths[idx]:.0f}nm: VIP={vip_scores[idx]:.3f}")
    
print(f"\n💡 FOR THIS TECATOR DATA:")
print(f"   • VIP > 1 indicates wavelength contributes significantly to model")
print(f"   • {important_vip} wavelengths out of 100 are truly important")
print(f"   • This explains why PLSR works: it identifies and uses key wavelengths")
print(f"   • In meat analysis, these likely correspond to")
print(f"     fat-specific absorption bands (C-H, C=O bonds)")


VIP SCORE INTERPRETATION

Wavelengths with VIP > 1: 38/100
  (38.0% of wavelengths are 'important')

Top 5 most important wavelengths for fat prediction:
  λ=931nm: VIP=1.594
  λ=929nm: VIP=1.580
  λ=933nm: VIP=1.546
  λ=927nm: VIP=1.526
  λ=850nm: VIP=1.446

💡 FOR THIS TECATOR DATA:
   • VIP > 1 indicates wavelength contributes significantly to model
   • 38 wavelengths out of 100 are truly important
   • This explains why PLSR works: it identifies and uses key wavelengths
   • In meat analysis, these likely correspond to
     fat-specific absorption bands (C-H, C=O bonds)


### 4.10 ⭐⭐⭐ **NEW: Hotelling T²** - Outlier Detection

Hotelling T² identifies outliers in the PLS score space (X-space).

In [76]:
# Compute Hotelling T² for TRAINING set
# Use PLS score variances (NOT PCA eigenvalues, which correspond to a different decomposition)
score_variances = np.var(T_pls, axis=0, ddof=1)
T2 = np.sum(T_pls**2 / score_variances, axis=1)

# Critical values based on training set size
n = len(X_train_std)
k = optimal_k_pls
from scipy.stats import f

F_crit_95 = f.ppf(0.95, k, n-k)
T2_95 = (k * (n-1) / (n-k)) * F_crit_95

F_crit_99 = f.ppf(0.99, k, n-k)
T2_99 = (k * (n-1) / (n-k)) * F_crit_99

print("="*70)
print("HOTELLING T² - OUTLIER DETECTION (Training Set)")
print("="*70)
print(f"\nT² control limits (based on {n} training samples, {k} components):")
print(f"  95% limit: {T2_95:.2f}")
print(f"  99% limit: {T2_99:.2f}")
print(f"\nOutliers detected:")
print(f"  Above 95%: {np.sum(T2 > T2_95)} samples ({np.sum(T2 > T2_95)/n*100:.1f}%)")
print(f"  Above 99%: {np.sum(T2 > T2_99)} samples ({np.sum(T2 > T2_99)/n*100:.1f}%)")
print(f"\n💡 INTERPRETATION:")
print(f"   • Hotelling T² measures distance from PLS model center")
print(f"   • High T² = unusual spectral pattern (may be interesting samples!)")
print(f"   • In meat analysis, might indicate atypical fat distribution")

# Plot
fig = go.Figure()

colors_t2 = ['green' if t < T2_95 else 'orange' if t < T2_99 else 'red' for t in T2]

fig.add_trace(go.Scatter(
    x=np.arange(len(T2)),
    y=T2,
    mode='markers',
    marker=dict(size=8, color=colors_t2),
    text=[f'Sample {i}<br>T²={T2[i]:.2f}' for i in range(len(T2))],
    hovertemplate='%{text}<extra></extra>'
))

fig.add_hline(y=T2_95, line_dash="dash", line_color="orange",
              annotation_text="95% limit")
fig.add_hline(y=T2_99, line_dash="dash", line_color="red",
              annotation_text="99% limit")

fig.update_layout(
    title="⭐ Hotelling T² - Outliers in Score Space",
    xaxis_title="Sample Index",
    yaxis_title="Hotelling T²",
    width=1000,
    height=500
)

fig.show()

print("\n💡 Interpretation:")
print("  Green: Normal samples")
print("  Orange: Moderate outliers (95-99%)")
print("  Red: Extreme outliers (>99%)")

HOTELLING T² - OUTLIER DETECTION (Training Set)

T² control limits (based on 172 training samples, 13 components):
  95% limit: 24.92
  99% limit: 31.37

Outliers detected:
  Above 95%: 20 samples (11.6%)
  Above 99%: 10 samples (5.8%)

💡 INTERPRETATION:
   • Hotelling T² measures distance from PLS model center
   • High T² = unusual spectral pattern (may be interesting samples!)
   • In meat analysis, might indicate atypical fat distribution



💡 Interpretation:
  Green: Normal samples
  Orange: Moderate outliers (95-99%)
  Red: Extreme outliers (>99%)


### 4.11 ⭐⭐⭐ **NEW: Q Residuals (DModX)** - Model Fit

Q residuals measure how well samples are described by the model.

In [77]:
# Compute Q residuals
X_reconstructed = T_pls @ P_pls.T
X_residuals = X_train_std - X_reconstructed
Q = np.sum(X_residuals**2, axis=1)

# Critical value - Jackson-Mudholkar approximation using residual eigenvalues
eigenvalues_residual = np.linalg.eigvalsh(np.cov(X_residuals.T))
eigenvalues_residual = eigenvalues_residual[eigenvalues_residual > 0]

theta1 = np.sum(eigenvalues_residual)
theta2 = np.sum(eigenvalues_residual**2)
theta3 = np.sum(eigenvalues_residual**3)
h0 = 1 - (2 * theta1 * theta3) / (3 * theta2**2)

z_alpha = stats.norm.ppf(0.95)
Q_crit_95 = theta1 * (z_alpha * h0 * np.sqrt(2 * theta2) / theta1 + 1 + theta2 * h0 * (h0 - 1) / theta1**2)**(1/h0)

print("="*70)
print("Q RESIDUALS (DModX) - MODEL FIT")
print("="*70)
print(f"\nQ critical (95%): {Q_crit_95:.6f}")
print(f"Q mean: {np.mean(Q):.6f}, Q max: {np.max(Q):.6f}")
print(f"Samples exceeding: {np.sum(Q > Q_crit_95)} / {len(Q)}")

# Plot
fig = go.Figure()

colors_q = ['green' if q < Q_crit_95 else 'red' for q in Q]

fig.add_trace(go.Scatter(
    x=np.arange(len(Q)),
    y=Q,
    mode='markers',
    marker=dict(size=8, color=colors_q),
    text=[f'Sample {i}<br>Q={Q[i]:.2f}' for i in range(len(Q))],
    hovertemplate='%{text}<extra></extra>'
))

fig.add_hline(y=Q_crit_95, line_dash="dash", line_color="red",
              annotation_text="95% limit (DCrit)")

fig.update_layout(
    title="⭐ Q Residuals (DModX) - Distance to Model",
    xaxis_title="Sample Index",
    yaxis_title="Q Residual",
    width=1000,
    height=500
)

fig.show()

print("\n💡 Interpretation:")
print("  Green: Well described by model")
print("  Red: Poorly described (different structure or errors)")

Q RESIDUALS (DModX) - MODEL FIT

Q critical (95%): 0.000023
Q mean: 0.000009, Q max: 0.000110
Samples exceeding: 20 / 172



💡 Interpretation:
  Green: Well described by model
  Red: Poorly described (different structure or errors)


### 4.12 ⭐⭐ **NEW: T² vs Q Plot**

Combined view of outlier types.

In [78]:
fig = go.Figure()

# Color by quadrant
colors_combo = []
for t, q in zip(T2, Q):
    if t <= T2_95 and q <= Q_crit_95:
        colors_combo.append('green')  # Normal
    elif t > T2_95 and q <= Q_crit_95:
        colors_combo.append('orange')  # High T² only
    elif t <= T2_95 and q > Q_crit_95:
        colors_combo.append('red')  # High Q only
    else:
        colors_combo.append('darkred')  # Both high!

fig.add_trace(go.Scatter(
    x=T2, y=Q,
    mode='markers',
    marker=dict(size=8, color=colors_combo),
    text=[f'Sample {i}<br>T²={T2[i]:.2f}<br>Q={Q[i]:.2f}' for i in range(len(T2))],
    hovertemplate='%{text}<extra></extra>'
))

fig.add_vline(x=T2_95, line_dash="dash", line_color="red")
fig.add_hline(y=Q_crit_95, line_dash="dash", line_color="red")

# Add quadrant labels
fig.add_annotation(x=T2_95/2, y=Q_crit_95*2, text="High Q<br>(poor fit)",
    showarrow=False, font=dict(size=10))
fig.add_annotation(x=T2_95*2, y=Q_crit_95/2, text="High T²<br>(outlier)",
    showarrow=False, font=dict(size=10))
fig.add_annotation(x=T2_95*2, y=Q_crit_95*2, text="Both<br>(REVIEW!)",
    showarrow=False, font=dict(size=10))

fig.update_layout(
    title="T² vs Q - Outlier Classification",
    xaxis_title="Hotelling T²",
    yaxis_title="Q Residual",
    width=800,
    height=600
)

fig.show()

### 4.13 ⭐⭐⭐ **NEW: Williams Plot** - Applicability Domain

The Williams plot shows prediction reliability.

In [79]:
# Compute leverage
leverage = np.sum(T_pls @ np.linalg.inv(T_pls.T @ T_pls) * T_pls, axis=1)

# Standardized residuals
residuals_cv = y_train - y_train_pred_pls_cv
std_residuals = (residuals_cv - np.mean(residuals_cv)) / np.std(residuals_cv)

# Critical values
h_crit = 3 * (optimal_k_pls + 1) / len(y_train)
r_crit = 3

print("="*70)
print("WILLIAMS PLOT - APPLICABILITY DOMAIN")
print("="*70)
print(f"\nCritical values:")
print(f"  Leverage: {h_crit:.4f}")
print(f"  Residual: ±{r_crit}")
print(f"\nSamples:")
print(f"  High leverage: {np.sum(leverage > h_crit)}")
print(f"  High residual: {np.sum(np.abs(std_residuals) > r_crit)}")

# Plot
fig = go.Figure()

colors_w = []
for h, r in zip(leverage, std_residuals):
    if h <= h_crit and abs(r) <= r_crit:
        colors_w.append('green')
    elif h > h_crit and abs(r) <= r_crit:
        colors_w.append('orange')
    elif h <= h_crit and abs(r) > r_crit:
        colors_w.append('red')
    else:
        colors_w.append('darkred')

fig.add_trace(go.Scatter(
    x=leverage, y=std_residuals,
    mode='markers',
    marker=dict(size=8, color=colors_w),
    text=[f'Sample {i}<br>h={leverage[i]:.3f}<br>r={std_residuals[i]:.2f}' 
          for i in range(len(leverage))],
    hovertemplate='%{text}<extra></extra>'
))

fig.add_vline(x=h_crit, line_dash="dash", line_color="red",
              annotation_text="Leverage limit")
fig.add_hline(y=r_crit, line_dash="dash", line_color="red")
fig.add_hline(y=-r_crit, line_dash="dash", line_color="red")

# Warning zone
fig.add_shape(type="rect",
    x0=h_crit, y0=-r_crit, x1=max(leverage)*1.1, y1=r_crit,
    line=dict(color="orange", width=2, dash="dot"),
    fillcolor="yellow", opacity=0.1
)

fig.update_layout(
    title="⭐ Williams Plot - Prediction Reliability",
    xaxis_title="Leverage (h)",
    yaxis_title="Standardized Residual",
    width=800,
    height=600
)

fig.show()

print("\n💡 Interpretation:")
print("  Green: Reliable predictions")
print("  Orange: High leverage, good fit (influential)")
print("  Red: Outliers (poor fit)")
print("  Dark red: High leverage + outlier (DANGEROUS!)")

WILLIAMS PLOT - APPLICABILITY DOMAIN

Critical values:
  Leverage: 0.2442
  Residual: ±3

Samples:
  High leverage: 7
  High residual: 1



💡 Interpretation:
  Green: Reliable predictions
  Orange: High leverage, good fit (influential)
  Red: Outliers (poor fit)
  Dark red: High leverage + outlier (DANGEROUS!)


### 4.14 ⭐⭐ **NEW: R² Train vs CV** - Overfitting Check

In [80]:
# Compute R² for different k
r2_train_list = []
r2_cv_list = []

for n_comp in range(1, max_components+1):
    pls_temp = PLSRegression(n_components=n_comp, scale=False)
    pls_temp.fit(X_train_std, y_train_std)
    y_pred_train = pls_temp.predict(X_train_std).ravel()
    r2_train_list.append(r2_score(y_train_std, y_pred_train))
    r2_cv_list.append(1 - pls_results[n_comp]['cv_rmse_mean']**2)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_values, y=r2_train_list,
    mode='lines+markers', name='Training R²',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=k_values, y=r2_cv_list,
    mode='lines+markers', name='CV R²',
    line=dict(color='red', width=2)
))

fig.add_vline(x=optimal_k_pls, line_dash="dash", line_color="green",
              annotation_text=f"Optimal k={optimal_k_pls}")

fig.update_layout(
    title="⭐ R² Train vs CV - Overfitting Detection",
    xaxis_title="Number of Components",
    yaxis_title="R²",
    yaxis_range=[0, 1],
    width=1000,
    height=500
)

fig.show()

gap = r2_train_list[optimal_k_pls-1] - r2_cv_list[optimal_k_pls-1]
print(f"\nR² gap at optimal k: {gap:.4f}")
if gap > 0.1:
    print("⚠️  Large gap indicates overfitting!")
else:
    print("✅ Small gap indicates good generalization")


R² gap at optimal k: 0.0156
✅ Small gap indicates good generalization


### 4.15 ⭐⭐ **NEW: Explained Variance (X and Y)**

In [81]:
# Variance in X (training set)
var_X_explained = []
for i in range(optimal_k_pls):
    X_recon = T_pls[:, :i+1] @ P_pls[:, :i+1].T
    var_X_explained.append((1 - np.var(X_train_std - X_recon) / np.var(X_train_std)) * 100)

# Variance in Y (training set)
var_Y_explained = []
for i in range(optimal_k_pls):
    pls_temp = PLSRegression(n_components=i+1, scale=False)
    pls_temp.fit(X_train_std, y_train_std)
    y_pred = pls_temp.predict(X_train_std).ravel()
    var_Y_explained.append(r2_score(y_train_std, y_pred) * 100)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Variance in X", "Variance in Y"))

fig.add_trace(go.Bar(x=list(range(1, optimal_k_pls+1)), y=var_X_explained,
    marker_color='lightblue'), row=1, col=1)

fig.add_trace(go.Bar(x=list(range(1, optimal_k_pls+1)), y=var_Y_explained,
    marker_color='lightcoral'), row=1, col=2)

fig.update_xaxes(title_text="Component", row=1, col=1)
fig.update_xaxes(title_text="Component", row=1, col=2)
fig.update_yaxes(title_text="Variance (%)", row=1, col=1)
fig.update_yaxes(title_text="Variance (%)", row=1, col=2)

fig.update_layout(
    title="⭐ Variance Explained by PLS Components",
    width=1200, height=500,
    showlegend=False
)

fig.show()

print(f"\nTotal variance explained:")
print(f"  X: {var_X_explained[-1]:.2f}%")
print(f"  Y: {var_Y_explained[-1]:.2f}%")


Total variance explained:
  X: 100.00%
  Y: 97.68%


### 4.16 ⭐ **NEW: Q-Q Plot** - Residual Normality

In [82]:
residuals_sorted = np.sort(residuals_cv)
theoretical_q = stats.norm.ppf(np.linspace(0.01, 0.99, len(residuals_cv)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=theoretical_q, y=residuals_sorted,
    mode='markers',
    marker=dict(size=6, color='blue')
))

# Reference line
line_x = [theoretical_q.min(), theoretical_q.max()]
line_y = [residuals_sorted.min(), residuals_sorted.max()]
fig.add_trace(go.Scatter(
    x=line_x, y=line_y,
    mode='lines',
    line=dict(color='red', dash='dash', width=2),
    name='Normal'
))

fig.update_layout(
    title="Q-Q Plot - Check Residual Normality",
    xaxis_title="Theoretical Quantiles",
    yaxis_title="Sample Quantiles",
    width=700, height=700
)

fig.show()

shapiro_stat, shapiro_p = stats.shapiro(residuals_cv)
print(f"\nShapiro-Wilk test: p-value = {shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("✅ Residuals appear normally distributed")
else:
    print("⚠️  Residuals may not be normally distributed")


Shapiro-Wilk test: p-value = 0.0000
⚠️  Residuals may not be normally distributed


### 4.17 ⭐⭐ **NEW: Inner Relation Plot** - PLS Validity

In [83]:
if optimal_k_pls >= 2:
    u_pls = pls_optimal.y_scores_
    
    fig = make_subplots(
        rows=1, cols=min(2, optimal_k_pls),
        subplot_titles=[f"Component {i+1}" for i in range(min(2, optimal_k_pls))]
    )
    
    for i in range(min(2, optimal_k_pls)):
        # Scatter
        fig.add_trace(go.Scatter(
            x=T_pls[:, i], y=u_pls[:, i],
            mode='markers',
            marker=dict(size=6, color='blue'),
            showlegend=False
        ), row=1, col=i+1)
        
        # Linear fit
        slope, intercept = np.polyfit(T_pls[:, i], u_pls[:, i], 1)
        x_line = np.array([T_pls[:, i].min(), T_pls[:, i].max()])
        y_line = slope * x_line + intercept
        
        fig.add_trace(go.Scatter(
            x=x_line, y=y_line,
            mode='lines',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False
        ), row=1, col=i+1)
        
        fig.update_xaxes(title_text=f"X-score t{i+1}", row=1, col=i+1)
        fig.update_yaxes(title_text=f"Y-score u{i+1}", row=1, col=i+1)
    
    fig.update_layout(
        title="⭐ Inner Relation - Check PLS Linearity",
        width=1000, height=500
    )
    
    fig.show()
    
    print("💡 Points should follow red line closely")
    print("   R² > 0.9 indicates good linear inner relation")
else:
    print("⚠️  Need ≥2 components")

💡 Points should follow red line closely
   R² > 0.9 indicates good linear inner relation


---

## **Section 5: PLS2 - Multiple Responses**

### 5.1 Create Multiple Response Matrix

In [84]:
# Use REAL Tecator responses from the properly split data
# Y_all_train contains [Fat, Moisture, Protein] for training samples
response_names = ['Fat', 'Moisture', 'Protein']

print("="*70)
print("PLS2: MULTIPLE RESPONSES")
print("="*70)
print(f"\nY_all_train shape: {Y_all_train.shape} (training samples)")
print(f"Y_all_test shape: {Y_all_test.shape} (test samples)")

for i, name in enumerate(response_names):
    print(f"\n{name} (Training):")
    print(f"  Mean: {Y_all_train[:, i].mean():.2f}%")
    print(f"  Range: [{Y_all_train[:, i].min():.2f}, {Y_all_train[:, i].max():.2f}]")

corr_Y = np.corrcoef(Y_all_train.T)
print(f"\nResponse correlations (Training Set):")
print(pd.DataFrame(corr_Y, index=response_names, columns=response_names).round(3))

print(f"\n💡 Using REAL multi-response data from Tecator dataset.")
print(f"   (Not synthetic data!)")

PLS2: MULTIPLE RESPONSES

Y_all_train shape: (172, 3) (training samples)
Y_all_test shape: (43, 3) (test samples)

Fat (Training):
  Mean: 18.05%
  Range: [0.90, 49.10]

Moisture (Training):
  Mean: 63.26%
  Range: [39.30, 76.60]

Protein (Training):
  Mean: 17.71%
  Range: [11.00, 21.80]

Response correlations (Training Set):
            Fat  Moisture  Protein
Fat       1.000    -0.988   -0.859
Moisture -0.988     1.000    0.814
Protein  -0.859     0.814    1.000

💡 Using REAL multi-response data from Tecator dataset.
   (Not synthetic data!)


### 5.2 PLS2 Cross-Validation

In [85]:
# Standardize multi-response using TRAINING data only
scaler_Y_multi = StandardScaler()
Y_train_multi_std = scaler_Y_multi.fit_transform(Y_all_train)  # Fit on train
Y_test_multi_std = scaler_Y_multi.transform(Y_all_test)  # Transform test

max_comp_pls2 = 20
print("Running PLS2 CV on TRAINING set...")

pls2_results = {}
for n_comp in range(1, max_comp_pls2+1):
    cv_scores = []
    for train_idx, val_idx in cv.split(X_train_std):
        X_cv_train, X_cv_val = X_train_std[train_idx], X_train_std[val_idx]
        Y_cv_train, Y_cv_val = Y_train_multi_std[train_idx], Y_train_multi_std[val_idx]
        pls2 = PLSRegression(n_components=n_comp, scale=False)
        pls2.fit(X_cv_train, Y_cv_train)
        Y_pred = pls2.predict(X_cv_val)
        cv_scores.append(mean_squared_error(Y_cv_val, Y_pred))
    pls2_results[n_comp] = {
        'cv_rmse_mean': np.sqrt(np.mean(cv_scores)),
        'cv_rmse_std': np.std(np.sqrt(cv_scores))
    }

rmse_values_pls2 = [pls2_results[k]['cv_rmse_mean'] for k in range(1, max_comp_pls2+1)]
optimal_k_pls2 = np.argmin(rmse_values_pls2) + 1

print(f"✅ Optimal: k={optimal_k_pls2}, RMSE={rmse_values_pls2[optimal_k_pls2-1]:.4f}")

Running PLS2 CV on TRAINING set...
✅ Optimal: k=14, RMSE=0.2153


### 5.3 Fit PLS2 Model

In [86]:
pls2_optimal = PLSRegression(n_components=optimal_k_pls2, scale=False)
pls2_optimal.fit(X_train_std, Y_train_multi_std)

Y_train_pred_pls2_std = pls2_optimal.predict(X_train_std)
Y_train_pred_pls2 = scaler_Y_multi.inverse_transform(Y_train_pred_pls2_std)

# CV predictions on training set
Y_train_pred_pls2_cv_std = np.zeros_like(Y_train_multi_std)
for train_idx, val_idx in cv.split(X_train_std):
    X_cv_train, X_cv_val = X_train_std[train_idx], X_train_std[val_idx]
    Y_cv_train, Y_cv_val = Y_train_multi_std[train_idx], Y_train_multi_std[val_idx]
    pls2_cv = PLSRegression(n_components=optimal_k_pls2, scale=False)
    pls2_cv.fit(X_cv_train, Y_cv_train)
    Y_train_pred_pls2_cv_std[val_idx] = pls2_cv.predict(X_cv_val)

Y_train_pred_pls2_cv = scaler_Y_multi.inverse_transform(Y_train_pred_pls2_cv_std)

print(f"PLS2 (k={optimal_k_pls2}) CV Performance on TRAINING set:")
for i, name in enumerate(response_names):
    r2 = r2_score(Y_all_train[:, i], Y_train_pred_pls2_cv[:, i])
    rmse = np.sqrt(mean_squared_error(Y_all_train[:, i], Y_train_pred_pls2_cv[:, i]))
    print(f"\n{name}:")
    print(f"  R²: {r2:.4f}")
    print(f"  RMSE: {rmse:.4f}%")

PLS2 (k=14) CV Performance on TRAINING set:

Fat:
  R²: 0.9607
  RMSE: 2.4964%

Moisture:
  R²: 0.9532
  RMSE: 2.1196%

Protein:
  R²: 0.9460
  RMSE: 0.6881%


### 5.4 PLS2 Predictions

In [87]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=[f"{name}: Actual vs Predicted (CV)" for name in response_names])

colors = ['blue', 'green', 'orange']

for i, (name, color) in enumerate(zip(response_names, colors)):
    fig.add_trace(go.Scatter(
        x=Y_all_train[:, i], y=Y_train_pred_pls2_cv[:, i],
        mode='markers', marker=dict(color=color, size=6, opacity=0.6),
        showlegend=False), row=1, col=i+1)
    
    y_range = [Y_all_train[:, i].min(), Y_all_train[:, i].max()]
    fig.add_trace(go.Scatter(
        x=y_range, y=y_range,
        mode='lines', line=dict(color='black', dash='dash'),
        showlegend=False), row=1, col=i+1)
    
    fig.update_xaxes(title_text=f"Actual {name} (%)", row=1, col=i+1)
    fig.update_yaxes(title_text=f"Predicted (%)", row=1, col=i+1)

fig.update_layout(title=f"PLS2 Predictions (k={optimal_k_pls2}) - Training Set CV",
    width=1400, height=450)
fig.show()

---

## **Section 6: Comprehensive Comparison**

### 6.1 Coefficient Comparison

In [88]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=wavelengths, y=coef_mlr,
    mode='lines', name='OLS',
    line=dict(color='red', width=1), opacity=0.3))

fig.add_trace(go.Scatter(x=wavelengths, y=theta_pcr,
    mode='lines', name=f'PCR (k={optimal_k_pcr})',
    line=dict(color='blue', width=2), opacity=0.6))

fig.add_trace(go.Scatter(x=wavelengths, y=coef_pls,
    mode='lines', name=f'PLS1 (k={optimal_k_pls})',
    line=dict(color='green', width=2.5)))

coef_pls2_fat = pls2_optimal.coef_[:, 0]
fig.add_trace(go.Scatter(x=wavelengths, y=coef_pls2_fat,
    mode='lines', name=f'PLS2 (k={optimal_k_pls2})',
    line=dict(color='purple', width=2.5, dash='dot')))

fig.add_hline(y=0, line_dash="dash", line_color="gray")

fig.update_layout(
    title="Regression Coefficients: All Methods",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Coefficient Value",
    width=1200, height=600,
    hovermode='x unified'
)

fig.show()

print("\n📊 Coefficient L2 Norms:")
print(f"  OLS:  {np.linalg.norm(coef_mlr):.4f}")
print(f"  PCR:  {np.linalg.norm(theta_pcr):.4f}")
print(f"  PLS1: {np.linalg.norm(coef_pls):.4f}")
print(f"  PLS2: {np.linalg.norm(coef_pls2_fat):.4f}")


📊 Coefficient L2 Norms:
  OLS:  8076.7622
  PCR:  114.8081
  PLS1: 127.8560
  PLS2: 8.1733


### 6.2 Performance Comparison

In [89]:
methods = ['OLS', 'PCR', 'PLS1', 'PLS2 (Fat)']
r2_values = [
    cv_r2_mlr.mean(),
    r2_pcr_cv,
    r2_pls_cv,
    r2_score(Y_all_train[:, 0], Y_train_pred_pls2_cv[:, 0])
]
components = [100, optimal_k_pcr, optimal_k_pls, optimal_k_pls2]
rmse_values = [
    cv_rmse_mlr.mean(),
    rmse_pcr_cv,
    rmse_pls_cv,
    np.sqrt(mean_squared_error(Y_all_train[:, 0], Y_train_pred_pls2_cv[:, 0]))
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=methods, y=r2_values,
    marker_color=['red', 'blue', 'green', 'purple'],
    text=[f'{r2:.4f}' for r2 in r2_values],
    textposition='outside'
))

fig.update_layout(
    title="R² Comparison (Cross-Validation)",
    xaxis_title="Method",
    yaxis_title="R² (CV)",
    yaxis_range=[0, 1],
    width=800, height=500
)

fig.show()

### 6.3 Summary Table

In [90]:
comparison_data = {
    'Method': methods,
    'Components': components,
    'CV R²': r2_values,
    'CV RMSE (%)': rmse_values
}

df_comparison = pd.DataFrame(comparison_data)

print("="*70)
print("FINAL PERFORMANCE COMPARISON")
print("="*70)
print("\n" + df_comparison.to_string(index=False))

print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)

improvement = (r2_pls_cv - cv_r2_mlr.mean()) * 100

print(f"\n1. OLS FAILURE:")
print(f"   CV R² = {cv_r2_mlr.mean():.4f}")
print(f"   Severe multicollinearity")

print(f"\n2. PCR IMPROVEMENT:")
print(f"   k = {optimal_k_pcr}, R² = {r2_pcr_cv:.4f}")

print(f"\n3. PLS SUPERIORITY:")
print(f"   k = {optimal_k_pls} ({optimal_k_pcr-optimal_k_pls} fewer!)")
print(f"   R² = {r2_pls_cv:.4f} ⭐ BEST")
print(f"   Improvement: {improvement:.1f}%")

print(f"\n4. PLS2 EFFICIENCY:")
print(f"   k = {optimal_k_pls2}")
print(f"   Handles 3 responses efficiently")

FINAL PERFORMANCE COMPARISON

    Method  Components    CV R²  CV RMSE (%)
       OLS         100 0.939626     0.233462
       PCR          17 0.959271     2.540880
      PLS1          13 0.960912     2.489163
PLS2 (Fat)          14 0.960684     2.496427

KEY FINDINGS

1. OLS FAILURE:
   CV R² = 0.9396
   Severe multicollinearity

2. PCR IMPROVEMENT:
   k = 17, R² = 0.9593

3. PLS SUPERIORITY:
   k = 13 (4 fewer!)
   R² = 0.9609 ⭐ BEST
   Improvement: 2.1%

4. PLS2 EFFICIENCY:
   k = 14
   Handles 3 responses efficiently


## 🎯 Test Set Evaluation

Now we evaluate our models on the **held-out test set** to assess true generalization performance. This is the critical final step in model validation - the test set was not used during training or cross-validation, providing an unbiased estimate of performance on new data.

We will:
1. Generate predictions on the test set for all models
2. Compute test set metrics (R², RMSE)
3. Compare training vs test performance to check for overfitting
4. Visualize actual vs predicted values on test data

In [91]:
### Step 1: Generate predictions on test set for all models

# OLS predictions on test set
y_test_pred_mlr = mlr.predict(X_test_std)

# PCR predictions on test set
X_test_pca = pca_optimal.transform(X_test_std)
y_test_pred_pcr = reg_pcr.predict(X_test_pca)

# PLS1 predictions on test set
y_test_pred_pls = pls_optimal.predict(X_test_std)

# PLS2 predictions on test set (all responses)
Y_test_pred_pls2_std = pls2_optimal.predict(X_test_std)
Y_test_pred_pls2 = scaler_Y_multi.inverse_transform(Y_test_pred_pls2_std)
y_test_pred_pls2_fat = Y_test_pred_pls2_std[:, 0]  # Fat prediction in standardized scale

print("✅ Test predictions generated for all models")

print(f"   Test set size: {len(y_test)} samples")
print(f"   PLS2 predictions shape: {Y_test_pred_pls2.shape} ({', '.join(response_names)})")
print(f"   OLS predictions shape: {y_test_pred_mlr.shape}")
print(f"   PLS1 predictions shape: {y_test_pred_pls.shape}")

print(f"   PCR predictions shape: {y_test_pred_pcr.shape}")

✅ Test predictions generated for all models
   Test set size: 43 samples
   PLS2 predictions shape: (43, 3) (Fat, Moisture, Protein)
   OLS predictions shape: (43,)
   PLS1 predictions shape: (43,)
   PCR predictions shape: (43,)


In [92]:
### Step 2: Compute test set metrics

# Flatten PLS predictions if needed (from 2D to 1D)
if y_test_pred_pls.ndim > 1:
    y_test_pred_pls = y_test_pred_pls.ravel()

# Test set performance (standardized scale)
test_r2_mlr = r2_score(y_test_std, y_test_pred_mlr)
test_rmse_mlr = np.sqrt(mean_squared_error(y_test_std, y_test_pred_mlr))

test_r2_pcr = r2_score(y_test_std, y_test_pred_pcr)
test_rmse_pcr = np.sqrt(mean_squared_error(y_test_std, y_test_pred_pcr))

test_r2_pls = r2_score(y_test_std, y_test_pred_pls)
test_rmse_pls = np.sqrt(mean_squared_error(y_test_std, y_test_pred_pls))

# PLS2 test performance (fat only, standardized)
test_r2_pls2 = r2_score(Y_test_multi_std[:, 0], y_test_pred_pls2_fat)
test_rmse_pls2 = np.sqrt(mean_squared_error(Y_test_multi_std[:, 0], y_test_pred_pls2_fat))

# Training set performance (for comparison)
train_r2_mlr = r2_score(y_train_std, y_train_pred_mlr_std)
train_rmse_mlr = np.sqrt(mean_squared_error(y_train_std, y_train_pred_mlr_std))

train_r2_pcr = r2_score(y_train_std, y_train_pred_pcr_std)
train_rmse_pcr = np.sqrt(mean_squared_error(y_train_std, y_train_pred_pcr_std))

train_r2_pls = r2_score(y_train_std, y_train_pred_pls_std)
train_rmse_pls = np.sqrt(mean_squared_error(y_train_std, y_train_pred_pls_std))

train_r2_pls2 = r2_score(Y_train_multi_std[:, 0], Y_train_pred_pls2_std[:, 0])
train_rmse_pls2 = np.sqrt(mean_squared_error(Y_train_multi_std[:, 0], Y_train_pred_pls2_std[:, 0]))

print("📊 Test Set Performance:\n")
print(f"OLS:  R² = {test_r2_mlr:.4f}, RMSE = {test_rmse_mlr:.4f}")
print(f"PCR:  R² = {test_r2_pcr:.4f}, RMSE = {test_rmse_pcr:.4f}")
print(f"PLS1: R² = {test_r2_pls:.4f}, RMSE = {test_rmse_pls:.4f}")

print(f"PLS2: R² = {test_r2_pls2:.4f}, RMSE = {test_rmse_pls2:.4f} (Fat only)")
print("\n" + "="*60)
print("\n🔍 Train vs Test Comparison:\n")
print(f"OLS:  Train R² = {train_r2_mlr:.4f} | Test R² = {test_r2_mlr:.4f} | Diff = {train_r2_mlr - test_r2_mlr:.4f}")
print(f"PCR:  Train R² = {train_r2_pcr:.4f} | Test R² = {test_r2_pcr:.4f} | Diff = {train_r2_pcr - test_r2_pcr:.4f}")
print(f"PLS1: Train R² = {train_r2_pls:.4f} | Test R² = {test_r2_pls:.4f} | Diff = {train_r2_pls - test_r2_pls:.4f}")
print(f"PLS2: Train R² = {train_r2_pls2:.4f} | Test R² = {test_r2_pls2:.4f} | Diff = {train_r2_pls2 - test_r2_pls2:.4f}")

# PLS2 all-response test evaluation
print("\n📊 PLS2 Test Set - All Responses:")
for i, name in enumerate(response_names):
    r2_i = r2_score(Y_test_multi_std[:, i], Y_test_pred_pls2_std[:, i])
    rmse_i = np.sqrt(mean_squared_error(Y_all_test[:, i], Y_test_pred_pls2[:, i]))
    print(f"  {name}: R² = {r2_i:.4f}, RMSE = {rmse_i:.4f}%")

📊 Test Set Performance:

OLS:  R² = 0.7621, RMSE = 0.5104
PCR:  R² = 0.9569, RMSE = 0.2173
PLS1: R² = 0.9572, RMSE = 0.2165
PLS2: R² = 0.9569, RMSE = 0.2171 (Fat only)


🔍 Train vs Test Comparison:

OLS:  Train R² = 0.9965 | Test R² = 0.7621 | Diff = 0.2344
PCR:  Train R² = 0.9739 | Test R² = 0.9569 | Diff = 0.0170
PLS1: Train R² = 0.9768 | Test R² = 0.9572 | Diff = 0.0196
PLS2: Train R² = 0.9772 | Test R² = 0.9569 | Diff = 0.0203

📊 PLS2 Test Set - All Responses:
  Fat: R² = 0.9569, RMSE = 2.7339%
  Moisture: R² = 0.9158, RMSE = 2.9568%
  Protein: R² = 0.9489, RMSE = 0.7226%


In [93]:
### Step 3: Visualize test set predictions

# Create subplots for each model
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=(
        'OLS',
        f'PCR (k={optimal_k_pcr})',
        f'PLS1 (k={optimal_k_pls})',
        f'PLS2 (k={optimal_k_pls2})'
    ),
    horizontal_spacing=0.08
)

# Add perfect prediction line
min_val = min(y_test_std.min(), y_test_pred_mlr.min(), y_test_pred_pcr.min(), y_test_pred_pls.min())
max_val = max(y_test_std.max(), y_test_pred_mlr.max(), y_test_pred_pcr.max(), y_test_pred_pls.max())

# OLS
fig.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
               mode='lines', line=dict(color='red', dash='dash'), 
               name='Perfect Prediction', showlegend=True),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=y_test_std, y=y_test_pred_mlr, mode='markers',
               marker=dict(size=8, color='blue', opacity=0.6),
               name=f'OLS (R²={test_r2_mlr:.3f})', showlegend=True),
    row=1, col=1
)

# PCR
fig.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
               mode='lines', line=dict(color='red', dash='dash'), 
               showlegend=False),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=y_test_std, y=y_test_pred_pcr, mode='markers',
               marker=dict(size=8, color='green', opacity=0.6),
               name=f'PCR (R²={test_r2_pcr:.3f})', showlegend=True),
    row=1, col=2
)

# PLS1
fig.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
               mode='lines', line=dict(color='red', dash='dash'), 
               showlegend=False),
    row=1, col=3
)
fig.add_trace(
    go.Scatter(x=y_test_std, y=y_test_pred_pls, mode='markers',
               marker=dict(size=8, color='purple', opacity=0.6),
               name=f'PLS1 (R²={test_r2_pls:.3f})', showlegend=True),
    row=1, col=3
)

# PLS2 (Fat response)
fig.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
               mode='lines', line=dict(color='red', dash='dash'), 
               showlegend=False),
    row=1, col=4
)
fig.add_trace(
    go.Scatter(x=Y_test_multi_std[:, 0], y=y_test_pred_pls2_fat, mode='markers',
               marker=dict(size=8, color='orange', opacity=0.6),
               name=f'PLS2 (R²={test_r2_pls2:.3f})', showlegend=True),
    row=1, col=4
)

# Update axes
for col in range(1, 5):
    fig.update_xaxes(title_text="Actual (std)", row=1, col=col)
fig.update_yaxes(title_text="Predicted (std)", row=1, col=1)

fig.update_layout(
    title_text="Test Set Performance: Actual vs Predicted (Fat Content)",
    height=400,
    width=1500,
    showlegend=True
)

fig.show()

In [94]:
### Step 4: Comprehensive Train-CV-Test Comparison

# Compute PLS2 CV metrics for fat (from training set)
cv_r2_pls2_fat = r2_score(Y_all_train[:, 0], Y_train_pred_pls2_cv[:, 0])
cv_rmse_pls2_fat = np.sqrt(mean_squared_error(Y_all_train[:, 0], Y_train_pred_pls2_cv[:, 0]))

# Inverse transform RMSE to original scale for interpretability
# Note: R² and RMSE here are computed in standardized space for consistency

# Create comprehensive comparison table
final_comparison = pd.DataFrame({
    'Model': [f'OLS ({X_train.shape[1]} feat)', f'PCR (k={optimal_k_pcr})', f'PLS1 (k={optimal_k_pls})', f'PLS2 (k={optimal_k_pls2})'],
    'Components': [X_train.shape[1], optimal_k_pcr, optimal_k_pls, optimal_k_pls2],
    'Train R²': [train_r2_mlr, train_r2_pcr, train_r2_pls, train_r2_pls2],
    'Train RMSE': [train_rmse_mlr, train_rmse_pcr, train_rmse_pls, train_rmse_pls2],
    'CV R²': [cv_r2_mlr.mean(), r2_pcr_cv, r2_pls_cv, cv_r2_pls2_fat],
    'CV RMSE': [cv_rmse_mlr.mean(), rmse_pcr_cv, rmse_pls_cv, cv_rmse_pls2_fat],
    'Test R²': [test_r2_mlr, test_r2_pcr, test_r2_pls, test_r2_pls2],
    'Test RMSE': [test_rmse_mlr, test_rmse_pcr, test_rmse_pls, test_rmse_pls2],
})

# Calculate generalization gaps
final_comparison['Train-Test Gap (R²)'] = final_comparison['Train R²'] - final_comparison['Test R²']
final_comparison['CV-Test Gap (R²)'] = final_comparison['CV R²'] - final_comparison['Test R²']

print("="*110)
print("📈 COMPREHENSIVE MODEL COMPARISON: Train vs Cross-Validation vs Test\n")
print(final_comparison.to_string(index=False))
print("="*110)

# Identify best model on test set
best_test_idx = final_comparison['Test R²'].idxmax()
best_model = final_comparison.loc[best_test_idx, 'Model']

print(f"\n🏆 Best Model on Test Set: {best_model}")
print(f"   Test R² = {final_comparison.loc[best_test_idx, 'Test R²']:.4f}")
print(f"   Test RMSE = {final_comparison.loc[best_test_idx, 'Test RMSE']:.4f}")

# Check for overfitting
print("\n🔍 Overfitting Analysis:")
for idx, row in final_comparison.iterrows():
    gap = row['Train-Test Gap (R²)']
    if gap < 0.01:
        status = "✅ Excellent generalization"
    elif gap < 0.05:
        status = "✅ Good generalization"
    elif gap < 0.10:
        status = "⚠️ Moderate overfitting"
    else:
        status = "❌ Significant overfitting"
    print(f"   {row['Model']}: Gap = {gap:.4f} → {status}")

📈 COMPREHENSIVE MODEL COMPARISON: Train vs Cross-Validation vs Test

         Model  Components  Train R²  Train RMSE    CV R²  CV RMSE  Test R²  Test RMSE  Train-Test Gap (R²)  CV-Test Gap (R²)
OLS (100 feat)         100  0.996485    0.059285 0.939626 0.233462 0.762118   0.510368             0.234368          0.177508
    PCR (k=17)          17  0.973882    0.161611 0.959271 2.540880 0.956886   0.217275             0.016995          0.002385
   PLS1 (k=13)          13  0.976823    0.152239 0.960912 2.489163 0.957211   0.216455             0.019612          0.003701
   PLS2 (k=14)          14  0.977194    0.151016 0.960684 2.496427 0.956938   0.217144             0.020256          0.003745

🏆 Best Model on Test Set: PLS1 (k=13)
   Test R² = 0.9572
   Test RMSE = 0.2165

🔍 Overfitting Analysis:
   OLS (100 feat): Gap = 0.2344 → ❌ Significant overfitting
   PCR (k=17): Gap = 0.0170 → ✅ Good generalization
   PLS1 (k=13): Gap = 0.0196 → ✅ Good generalization
   PLS2 (k=14): Gap = 0.0203 →

### 📝 Interpretation and Key Insights

#### Why Test Set Evaluation Matters:
- **Training performance** can be misleadingly high due to overfitting
- **Cross-validation** provides better estimates but still uses training data
- **Test set** offers the only unbiased estimate of generalization to new data

#### Expected Results:
Based on the methodology:
1. **PLS1** should perform best on test data (optimal bias-variance tradeoff)
2. **PCR** should show good generalization (dimensionality reduction helps)
3. **OLS** may show signs of overfitting (too many features, n < p initially)

#### What to Look For:
- **Small Train-Test gaps** (< 0.05 in R²) indicate good generalization
- **Large gaps** suggest overfitting to training data
- **Test R² close to CV R²** validates the cross-validation strategy
- **Consistent ranking** across CV and Test confirms model selection

#### Final Recommendation:
The model with the highest **test set R²** should be deployed for predicting fat content in new meat samples, as it has proven generalization ability.

---

## **🎉 Tutorial Complete!**

### **What You've Learned**

✅ **Data Exploration** (5 plots)
- Multicollinearity assessment
- Response distribution
- Wavelength correlations

✅ **OLS Analysis** (4 plots)
- Why it fails
- Coefficient instability
- Overfitting diagnostics

✅ **PCR Analysis** (4 plots)
- PCA variance explained
- PC-response correlations
- Cross-validation

✅ **PLS Comprehensive** (18 plots!) ⭐
- Score plots (2D & 3D)
- Standard loadings
- **Correlation loadings** (NEW!)
- **Correlation circle** (NEW!)
- VIP scores
- **Hotelling T²** (NEW!)
- **Q residuals / DModX** (NEW!)
- **T² vs Q plot** (NEW!)
- **Williams plot** (NEW!)
- **R² train vs CV** (NEW!)
- **Explained variance X & Y** (NEW!)
- **Q-Q plot** (NEW!)
- **Inner relation** (NEW!)

✅ **PLS2 Analysis** (3 plots)
- Multiple response modeling
- Efficiency demonstration

✅ **Final Comparison** (3 plots)
- All methods side-by-side
- Performance metrics

**Total: ~35 Interactive Plots!** 🎨

---

### **Key Takeaways**

1. **OLS fails** with multicollinearity → Use dimension reduction
2. **PCR works** but is unsupervised → May need many components
3. **PLS wins!** Supervised, fewer components, best accuracy
4. **Correlation loadings** are INDUSTRY STANDARD for interpretation
5. **Hotelling T² & Q residuals** essential for outlier detection
6. **Williams plot** defines applicability domain
7. **PLS2** efficiently handles multiple correlated responses

---

### **When to Use Each Method**

**OLS:** Low correlation, n >> p  
**PCR:** Need X-structure interpretation  
**PLS1:** High multicollinearity, single response ⭐ RECOMMENDED  
**PLS2:** Multiple correlated responses  

---

### **References**

1. Wold et al. (1983) - Original PLS
2. Geladi & Kowalski (1986) - PLS Tutorial
3. Martens & Næs (1989) - Multivariate Calibration
4. Eriksson et al. (2013) - Multi- and Megavariate Analysis

---

**Course:** TTK4260 - Multivariate Data Analysis and ML  
**Instructor:** Adil Rasheed, NTNU

**Questions?** adil.rasheed@ntnu.no

---

## 🚀 **What's Next?**

- Apply to your own data
- Try advanced variants (OPLS, Sparse PLS)
- Explore other methods (Ridge, Lasso, Elastic Net)
- Read the chemometrics literature

**Happy Learning!** 🎓